In [ ]:
%run ../globalvariables

In [ ]:
%run ../lakehousefunction

In [ ]:
dbutils.widgets.text("year_start", "2020")
dbutils.widgets.text("year_end", "2025")

YEAR_START = int(dbutils.widgets.get("year_start"))
YEAR_END = int(dbutils.widgets.get("year_end"))

In [ ]:
DATASET = "aire"
NOTEBOOK = "bronze/import_aire"

df_origins = pd.read_csv(ORIGIN_PATH)
df_datasets = pd.read_csv(DATASET_PATH)

row = df_datasets[df_datasets["dataset_destino"] == DATASET].iloc[0]
origin_row = df_origins[df_origins["id"] == row["origin"]].iloc[0]
CKAN_BASE = origin_row["endpoint"]

print(f"{DATASET}: origin={row['dataset_origin']} partition={row['partition']}")

In [ ]:
try:
    probe = fetch_with_retry(lambda: fetch_resources(row["dataset_origin"], CKAN_BASE))
    print(f"CKAN reachable - {len(probe)} resources")
except Exception as e:
    log_errors([error_record(NOTEBOOK, e)])
    print(f"connection failed: {e}")
    dbutils.notebook.exit(1)

In [ ]:
errors = []
ingested = []

resources = fetch_with_retry(lambda: fetch_resources(row["dataset_origin"], CKAN_BASE))

for key in partition_keys(row["partition"], YEAR_START, YEAR_END):
    try:
        resource = pick_resource(resources, row["partition"], key, row["file_type"], row["select_by"])
        df = read_resource(resource, row["file_type"], DATASET, key)
        df = sanitize_column_names(df)
        # Honour requested year range
        df = df.filter(f"CAST(ano AS INT) BETWEEN {YEAR_START} AND {YEAR_END}")
        df = apply_schema(df, DATASET)
        df = add_ingestion_metadata(
            df, source_system="CKAN", source_file=resource["url"],
            partition_key=key or "latest",
        )
        rows = df.count()

        replace_where = f"partition_key = '{key or 'latest'}'"
        if not write_bronze(df, DATASET, mode="overwrite", replace_where=replace_where):
            raise Exception("write_bronze returned False")

        ingested.append({
            "timestamp": datetime.now().isoformat(), "notebook_name": NOTEBOOK,
            "dataset_destino": DATASET, "partition_key": key or "latest",
            "rows_written": rows, "status": "SUCCESS",
        })
        print(f"ok {DATASET} {key or ''} ({rows} rows)")
    except ValueError as e:
        errors.append(error_record(NOTEBOOK, e, status="SKIPPED"))
        print(f"skip {DATASET} {key or ''}: not published yet - {e}")
    except Exception as e:
        errors.append(error_record(NOTEBOOK, e))
        print(f"fail {DATASET} {key or ''}: {type(e).__name__}: {e}")

In [ ]:
print(f"{DATASET}: {len(ingested)} partitions ok, {len(errors)} failed")
log_ingestion(ingested)
log_errors(errors)